In [13]:
import os
from openai import OpenAI
from dotenv import load_dotenv
import json 
from IPython.display import Markdown, display, update_display

In [14]:
# constants

MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'

In [15]:
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [16]:
# here is the question; type over this to ask something new

question = """
Analyze the following code snippet and explain it clearly.

Code:
yield from {book.get("author") for book in books if book.get("author")}

In your response, cover:
1. What the code does (high-level summary)
2. Step-by-step explanation of how it works
3. Explanation of key constructs used (e.g., yield from, set comprehension)
4. Why this approach might be used (benefits or intent)
5. Any edge cases, limitations, or potential issues
6. (Optional) Suggest alternative implementations if relevant

Keep the explanation concise but informative.
"""

In [17]:
system_prompt = """
You are an expert software engineering assistant.

Your job is to help users understand, summarize, explain, and reason about code, software architecture, and implementation details.

You can:
- Summarize source code clearly and accurately
- Explain what functions, classes, and modules do
- Describe control flow, dependencies, and interactions between components
- Identify assumptions, edge cases, and potential risks
- Explain code in beginner-friendly or expert-friendly language depending on the user’s request
- Help with debugging, refactoring suggestions, and design-level understanding
- Answer follow-up questions grounded in the provided code and context

Guidelines:
- Be precise, practical, and easy to understand
- Prioritize correctness over speculation
- If the code or context is incomplete, clearly state what is known and what is missing
- Do not invent behavior that is not supported by the code
- When summarizing, focus on purpose, inputs, outputs, key logic, and important dependencies
- When explaining code, use structured reasoning and reference specific parts of the code when helpful
- Highlight potential bugs, inefficiencies, security concerns, or maintainability issues when relevant
- Adapt the depth of explanation to the complexity of the code and the user's question

Output style:
- Default to concise but informative explanations
- Use bullet points or sections when they improve clarity
- Use simple language unless the user asks for a more technical explanation
- Where useful, include a short “high-level summary” followed by deeper details
"""

In [18]:
#Using GPT Models

In [19]:
def tutorial(question):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [20]:
tutorial(r"""
resource "google_iam_workload_identity_pool_provider" "wif-provider" {
  depends_on = [
    google_iam_workload_identity_pool.wif-pool
  ]

  workload_identity_pool_id          = google_iam_workload_identity_pool.wif-pool.workload_identity_pool_id
  workload_identity_pool_provider_id = var.workload-identity-pool-provider
  display_name                       = var.workload-identity-pool-provider
  project                            = var.project
  description                        = "Identity pool provider for GITHUB CI/CD Authentication"
  attribute_mapping = {
    "google.subject"       = "assertion.sub"
    "attribute.actor"      = "assertion.actor"
    "attribute.repository" = "assertion.repository"
  }
  attribute_condition = "assertion.repository_owner==\"${var.github_org}\""

  oidc {
    issuer_uri = "https://token.actions.githubusercontent.com"
  }
}
""")

This Terraform code resource defines a Google Cloud IAM Workload Identity Pool Provider, specifically configured for GitHub Actions integration to enable secure authentication in your CI/CD workflows.

### High-level summary:
- **Resource:** `google_iam_workload_identity_pool_provider`
- **Purpose:** Creates an OIDC identity provider in a Workload Identity Pool to allow GitHub Actions to authenticate to Google Cloud.
- **Use case:** Enables GitHub CI/CD workflows to access Google Cloud resources using Workload Identity Federation.

---

### Key parts and explanation:

- **depends_on:**  
  Ensures this provider is created only after the referenced `google_iam_workload_identity_pool.wif-pool` is created, guarding against ordering issues.

- **workload_identity_pool_id:**  
  Links this provider to an existing Workload Identity Pool (the pool acts like a container for identity providers).

- **workload_identity_pool_provider_id & display_name:**  
  Uniquely identifies this provider within the pool and sets a user-friendly display name — value taken from a variable (`var.workload-identity-pool-provider`).

- **project:**  
  The Google Cloud project where the pool and provider will be created (from `var.project`).

- **description:**  
  Describes this provider as for "GITHUB CI/CD Authentication" — this clarifies it's for integrating GitHub Actions with Google Cloud IAM.

- **attribute_mapping:**  
  Maps claims from the OIDC token received from GitHub Actions to attributes understood by Google IAM:
  - `"google.subject"` set from the OIDC token's `sub` claim.
  - `"attribute.actor"` from `actor` claim.
  - `"attribute.repository"` from `repository`.

- **attribute_condition:**  
  A condition to restrict identities that can authenticate via this provider. It checks that the claim `repository_owner` exactly matches the GitHub organization given by `var.github_org`. This restricts tokens to only those issued for repos owned by that GitHub organization.

- **oidc:**  
  Configures the OIDC settings for this provider:
  - `issuer_uri` is set to GitHub Actions' OIDC token issuer URL (`https://token.actions.githubusercontent.com`), meaning this provider accepts tokens issued by GitHub Actions.

---

### What happens overall:

When a GitHub Action workflow runs, it can request an OIDC token from the GitHub issuer (`token.actions.githubusercontent.com`). This token contains claims about the workflow, repo, and actor. Google Cloud uses this identity pool provider to accept this token, map the claims into a Google identity, and trust the request if it passes the condition (repo owner matches your org). This allows GitHub workflows to authenticate securely to Google Cloud without long-lived service account keys.

---

### Potential notes / risks:
- Make sure the variables (`var.workload-identity-pool-provider`, `var.project`, `var.github_org`) are set correctly.
- The attribute condition is string-equality on repo owner; more complex conditions might be needed for multi-org or finer-grained control.
- The `issuer_uri` must exactly match GitHub's OIDC provider URL.
- Make sure the referenced Workload Identity Pool (`google_iam_workload_identity_pool.wif-pool`) exists and is correctly configured.

---

If you want, I can explain how this fits with service accounts and role bindings to allow GitHub Actions to impersonate Google service accounts or provide a sample usage snippet.

In [21]:
#Using Ollama Model

In [22]:
def tutorial(question):
    OLLAMA_BASE_URL = "http://localhost:11434/v1"
    ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')
    stream = ollama.chat.completions.create(
        model="llama3.2",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [23]:
tutorial(r"""
resource "google_iam_workload_identity_pool_provider" "wif-provider" {
  depends_on = [
    google_iam_workload_identity_pool.wif-pool
  ]

  workload_identity_pool_id          = google_iam_workload_identity_pool.wif-pool.workload_identity_pool_id
  workload_identity_pool_provider_id = var.workload-identity-pool-provider
  display_name                       = var.workload-identity-pool-provider
  project                            = var.project
  description                        = "Identity pool provider for GITHUB CI/CD Authentication"
  attribute_mapping = {
    "google.subject"       = "assertion.sub"
    "attribute.actor"      = "assertion.actor"
    "attribute.repository" = "assertion.repository"
  }
  attribute_condition = "assertion.repository_owner==\"${var.github_org}\""

  oidc {
    issuer_uri = "https://token.actions.githubusercontent.com"
  }
}
""")

**High-Level Summary**

This Terraform configuration sets up a Google Cloud Identity Pool Provider (IP Provider) for authenticating with GitHub through Continuous Integration/Continuous Deployment (CI/CD) jobs. The IP Provider is linked to a workload identity pool and uses an OpenID Connect (OIDC) issuer to authenticate users.

**Key Components and Settings**

* **Workload Identity Pool**: `google_iam_workload_identity_pool.wif-pool` (dependent on the provider)
* **IP Provider**: `google_iam_workload_identity_pool_provider.wif-provider`
	+ Linked workload identity pool: `workload_identity_pool_id` is set to the ID of the linked workload identity pool.
	+ OIDC issuer: `oidc.issuer_uri` is set to `https://token.actions.githubusercontent.com`.
* **Attribute Mapping**: The provider maps specific attributes from the GitHub authentication response:
	+ `google.subject->attribute.actor`
	+ `attribute.repository`
* **Attribute Condition**: An attribute mapping condition specifies that the repository owner must be a member of the specified organization (`var.github_org`).
* **Project and Description**: The project and description are set to the values in the `project` and `description` variables, respectively.

**Understanding the Code**

This Terraform configuration uses the Google Cloud Resource Manager to create an IP Provider that integrates with a workload identity pool. The provider is configured to use OIDC authentication with GitHub.

Key dependencies include:

* The linked workload identity pool (`google_iam_workload_identity_pool.wif-pool`)
* The OIDC issuer URI
* Attribute mapping and attribute conditions are used to transform the authentication response from GitHub

**Potential Bugs or Issues**

While the code appears to be correct, a few potential issues should be considered:

* Make sure that the `var.github_org` variable is set correctly and contains the expected organization value.
* Verify that the workload identity pool and Google Cloud Identity Pool Provider are correctly linked.
* Ensure that the OIDC issuer URI remains valid and up-to-date.

**Refactoring Suggestions**

There are no immediate refactoring suggestions for this code snippet. However, consider making the `var` variables more descriptive and using a consistent naming convention (e.g., `github_org_variable` instead of just `github_org`).